# BuildCompiler SynBioHub Tutorial

This notebook shows how to authenticate to SynBioHub, load private collections into BuildCompiler, and run a full-build workflow using an abstract design URI.


In [ ]:
from getpass import getpass
import sbol2

from buildcompiler.api import BuildCompiler


In [ ]:
repository_url = "https://synbiohub.org"
user = "user"
password = getpass("SynBioHub password: ")

abstract_design_uri = "https://synbiohub.org/user/Gon/abstract_design/standard_GFP/1"
collections = [
    "https://synbiohub.org/user/Gon/impl_test/impl_test_collection/1",
    "https://synbiohub.org/user/Gon/Enzyme_Implementations/Enzyme_Implementations_collection/1",
]


In [ ]:
sbol_doc = sbol2.Document()

compiler = BuildCompiler.from_synbiohub(
    repository_url=repository_url,
    email=user,
    password=password,
    collections=collections,
    sbol_doc=sbol_doc,
)

print(f"Loaded SBOL objects: {len(list(sbol_doc))}")
print(f"Indexed plasmids: {len(getattr(compiler.inventory, 'plasmids', [])) if compiler.inventory else 'n/a'}")


In [ ]:
# Pull/resolve the abstract design into the shared document.
if compiler.repository_client is not None:
    compiler.repository_client.pull_identity(abstract_design_uri)

abstract_design = sbol_doc.find(abstract_design_uri)
if abstract_design is None:
    raise LookupError(f"Could not resolve abstract design: {abstract_design_uri}")

plan = compiler.plan([abstract_design])
result = compiler.execute(plan)
result


## Notes

- BuildCompiler reuses a single authenticated `sbol2.PartShop` client for collection pulls and identity fallback pulls.
- Tokens returned by SynBioHub login are kept in memory only; they are not serialized by default.
- Assembly uses Golden Gate digestion/ligation simulation and links reagent usages + generated products through a stage activity in SBOL.
